# 04. Machine Learning Agent — A Flexible CSV-Based ML Workflow

## Learning Objectives

- Set a data directory with `FilesystemBackend` and let the agent explore files freely
- Extend the `run_pandas` pattern from NB03 to create a `run_ml_code` tool with scikit-learn
- Let the agent explore data with built-in tools (`ls`, `read_file`, `glob`) and analyze it with `run_ml_code`
- Run EDA → preprocessing → model selection → training → evaluation through multi-turn conversation


## Overview

| Item | NB03 (Data Analysis) | NB04 (Machine Learning) |
|------|-------------------|----------------|
| **Backend** | `LocalShellBackend` | `FilesystemBackend` |
| **Data** | Sales CSV (8 rows) | User-provided CSV (demo: breast cancer, 569 rows) |
| **Custom tools** | `get_csv_path` + `run_pandas` | `run_ml_code` (adds sklearn) |
| **Built-in tools** | — | `ls`, `read_file`, `glob` (file exploration) |
| **Goal** | Aggregation, statistics, trend analysis | EDA → preprocessing → model training → comparison |


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"


In [ ]:
# Optional observability setup
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")


## NB03 vs. NB04: Extending the Backend and Tooling

In NB03, you used `LocalShellBackend` + `run_pandas` to execute pandas code.
In NB04, you extend that setup in two ways:

1. **Backend**: `FilesystemBackend(root_dir=DATA_DIR)` — the agent can freely explore the data directory with built-in tools such as `ls`, `read_file`, and `glob`
2. **Tool**: `run_ml_code` — adds `sklearn` to the execution namespace so the agent can run ML pipelines

```python
# NB03: LocalShellBackend + run_pandas
backend = LocalShellBackend(root_dir=tmp_dir, virtual_mode=True)
ns = {"pd": pd, "np": np, "csv_path": csv_path}

# NB04: FilesystemBackend + run_ml_code
backend = FilesystemBackend(root_dir=DATA_DIR, virtual_mode=True)
ns = {"pd": pd, "np": np, "sklearn": sklearn, "DATA_DIR": DATA_DIR}
```

> Because `FilesystemBackend` does not expose `execute`, it is safer than `LocalShellBackend`.


## Step 1: Set the Data Directory

If you change `DATA_DIR`, you can point the agent to **your own CSV data**.
The agent will inspect the directory with built-in tools such as `ls`, `glob`, and `read_file` to understand which files exist.

```python
# Example: point to your own data directory
DATA_DIR = "/path/to/your/data"
```


In [ ]:
import tempfile
import pandas as pd
from sklearn.datasets import load_breast_cancer

# ── Data directory setup ──────────────────────────────
# Change this to a directory that contains your own CSV files.
# For this demo, we save the breast_cancer dataset as a CSV.
DATA_DIR = tempfile.mkdtemp()

# Demo data creation (remove this block if you already have your own CSV)
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target
df.to_csv(os.path.join(DATA_DIR, "breast_cancer.csv"), index=False)

print(f"DATA_DIR: {DATA_DIR}")
print(f"Files: {os.listdir(DATA_DIR)}")


## Step 2: Create the `FilesystemBackend`

`FilesystemBackend` provides built-in file tools below the `root_dir`:

| Built-in Tool | Role |
|------------|------|
| `ls` | List directory contents |
| `read_file` | Read file contents |
| `glob` | Find files by pattern |
| `write_file` | Write files (for saving results) |

> `virtual_mode=True` prevents path escape patterns such as `..` or `~`.


In [ ]:
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(root_dir=DATA_DIR, virtual_mode=True)


## Step 3: Define `run_ml_code`

Extend the `run_pandas` pattern from NB03 by adding `sklearn` to the execution namespace.
Pass `DATA_DIR` into the namespace so the agent can load any CSV file inside the directory.

> Use built-in `ls` / `read_file` for file exploration and `run_ml_code` for code execution — keep the responsibilities separate.


In [ ]:
from langchain.tools import tool
import io, contextlib

@tool
def run_ml_code(code: str) -> str:
    """Execute sklearn/pandas Python code. Use print() to display results.
    Available modules: pd, np, sklearn, os. Access the data directory via DATA_DIR."""
    import pandas as pd, numpy as np, sklearn
    buf = io.StringIO()
    ns = {"pd": pd, "np": np, "sklearn": sklearn, "os": os, "DATA_DIR": DATA_DIR}
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, ns)
        return buf.getvalue() or "Execution finished (no printed output)"
    except Exception as e:
        return f"Error: {e}"


## Step 4: Create the Agent

The agent workflow is:
1. Explore CSV files inside `DATA_DIR` with built-in `ls` / `glob`
2. Preview data with built-in `read_file`
3. Run EDA → preprocessing → model training/comparison with `run_ml_code`

| Middleware | Role |
|---------|------|
| `ToolRetryMiddleware` | Automatically retries failed tool calls (up to 2 times) |
| `ModelCallLimitMiddleware` | Prevents infinite loops by limiting the run to 20 model calls |


In [ ]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import (
    ToolRetryMiddleware,
    ModelCallLimitMiddleware,
)
from prompts import ML_AGENT_PROMPT

ml_agent = create_deep_agent(
    model=model,
    tools=[run_ml_code],
    system_prompt=ML_AGENT_PROMPT,
    backend=backend,
    skills=["/skills/"],
    checkpointer=InMemorySaver(),
    middleware=[
        ToolRetryMiddleware(max_retries=2),
        ModelCallLimitMiddleware(run_limit=20),
    ],
)


## Step 5: File Exploration + EDA

Ask the agent to explore the data directory and analyze the dataset.
The agent can inspect the file list with built-in `ls` and then perform EDA with `run_ml_code`.


In [ ]:
thread = {"configurable": {"thread_id": "ml-1"}}

response = ml_agent.invoke(
    {"messages": [{"role": "user", "content": "Use ls to inspect the CSV files in the data directory, then use run_ml_code to load the data and analyze its shape, target distribution, and key summary statistics."}]},
    config={**thread, **lf_config},
)
print(response["messages"][-1].content)


## Step 6: Train and Compare Models

Ask the agent to train at least three suitable models and compare them with cross-validation.
The agent chooses the algorithms by itself.


In [ ]:
response = ml_agent.invoke(
    {"messages": [{"role": "user", "content": "Train at least three appropriate classification models and compare them with 5-fold cross-validation. Present the results as a table."}]},
    config={**thread, **lf_config},
)
print(response["messages"][-1].content)


## Step 7: Multi-Turn Follow-Up — Feature Importance

Reuse the same `thread_id` so the follow-up analysis keeps the earlier conversation context.


In [ ]:
response = ml_agent.invoke(
    {"messages": [{"role": "user", "content": "Analyze the feature importance of the best model. Show the top 10 features."}]},
    config={**thread, **lf_config},
)
print(response["messages"][-1].content)


## Step 8: Streaming — Additional Analysis

Observe the execution process in real time with `stream(subgraphs=True)`.


In [ ]:
chunks = []
for ns, chunk in ml_agent.stream(
    {"messages": [{"role": "user", "content": "Apply scaling to the data, compare the same models again, and summarize the performance difference before and after scaling in a table."}]},
    config={**thread, **lf_config},
    subgraphs=True,
):
    chunks.append((ns, chunk))

print(f"Received {len(chunks)} chunks in total", flush=True)
last = chunks[-1][1] if chunks else {}
if hasattr(last, "get") and last.get("messages"):
    print(last["messages"][-1].content, flush=True)


## Summary

| Item | Key Point |
|------|------|
| **Backend** | `FilesystemBackend(root_dir=DATA_DIR)` — point the agent at your data directory |
| **Built-in tools** | `ls`, `read_file`, `glob` — explore files |
| **Custom tool** | `run_ml_code` (pandas + numpy + sklearn) — execute ML code |
| **Workflow** | File exploration → EDA → preprocessing → model selection → cross-validation comparison |
| **Multi-turn** | `InMemorySaver` + same `thread_id` — preserve analysis context |

### Using Your Own Data

```python
# Just change DATA_DIR in Step 1
DATA_DIR = "/path/to/your/data"  # directory containing CSV files
```

The agent will explore files with `ls` and analyze them freely with `run_ml_code`.

---

**References:**
- `docs/deepagents/06-backends.md`
- `docs/deepagents/tutorials/data-analysis.md`
- [scikit-learn documentation](https://scikit-learn.org/stable/)

**Next Step:** → [05_deep_research_agent.ipynb](./05_deep_research_agent.ipynb): Build a deep research agent.
